# MiniSafeCoDe
**A small, readable re-implementation of the core idea behind SafeCoDe (Liu et al., 2025, arXiv:2509.19212): decode-time, context-aware safety steering for vision-language models.**

Runs on a free Colab T4 GPU. No training, no fine-tuning, one small open model (Qwen2-VL-2B).

**The two stages (simplified from the paper):**
1. **Contrastive decoding.** At every step, compare next-token logits with the *real* image against logits with a *Gaussian-noised* image. Tokens that depend on the visual content get boosted.
2. **Global-aware token modulation.** A judge pass asks the same model whether the request is unsafe *given the image*, then softly boosts or suppresses refusal tokens at the first generation step.

**Disclaimer:** this is a learning-sized sketch, not the official code. The paper uses a separate auxiliary judge and its own formulation details. Here the model judges itself and the formulas are the standard contrastive-decoding form.

Setup: `Runtime > Change runtime type > T4 GPU`, then run all cells.

In [ ]:
!pip -q install -U transformers accelerate

## 1. Load model

In [ ]:
import math, time, torch
from PIL import Image, ImageDraw
from transformers import (Qwen2VLForConditionalGeneration, AutoProcessor,
                          LogitsProcessor, LogitsProcessorList)

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
# small image budget keeps memory and latency low
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=128*28*28, max_pixels=384*28*28)
tok = processor.tokenizer
print("loaded on", model.device)

## 2. Helpers

In [ ]:
def build_inputs(image, text):
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=[prompt], images=[image], return_tensors="pt").to(model.device)
    inputs["pixel_values"] = inputs["pixel_values"].to(model.dtype)
    return inputs

# Token ids that start a refusal (first token of each phrase)
REFUSAL_PHRASES = ["I'm sorry", "I cannot", "I can't", "Sorry", "I apologize", "Unfortunately"]
REFUSAL_IDS = sorted({tok.encode(p, add_special_tokens=False)[0] for p in REFUSAL_PHRASES})
YES_ID = tok.encode("Yes", add_special_tokens=False)[0]
NO_ID = tok.encode("No", add_special_tokens=False)[0]

@torch.no_grad()
def judge_p_unsafe(image, question):
    """Stage 2 judge: P(unsafe | image, question) from the Yes/No logits."""
    q = (f"A user shows this image and asks: \"{question}\"\n"
         "Considering what is visible in the image, would answering this request "
         "be unsafe or harmful? Answer with Yes or No.")
    inputs = build_inputs(image, q)
    logits = model(**inputs).logits[0, -1].float()
    return torch.softmax(logits[[YES_ID, NO_ID]], dim=-1)[0].item()

## 3. The decoding processor (the core of the project)

At each step this replaces the normal logits with:

`score = (1 + beta) * logp_real_image  -  beta * logp_noisy_image`

then keeps only plausible tokens (within a factor `alpha` of the top token), and on the **first** step adds `gamma * (2 * p_unsafe - 1)` to refusal tokens.

In [ ]:
class MiniSafeCoDe(LogitsProcessor):
    def __init__(self, inputs, beta=1.0, sigma=1.0, alpha=0.1, p_unsafe=None, gamma=4.0, seed=0):
        self.beta, self.alpha, self.gamma, self.p_unsafe = beta, alpha, gamma, p_unsafe
        self.prompt_len = inputs["input_ids"].shape[1]
        self.grid = inputs["image_grid_thw"]
        pv = inputs["pixel_values"]
        g = torch.Generator(device=pv.device).manual_seed(seed)
        noise = torch.randn(pv.shape, generator=g, device=pv.device, dtype=torch.float32)
        self.noisy_pv = (pv.float() + sigma * noise).to(pv.dtype)   # Gaussian-noised image

    @torch.no_grad()
    def __call__(self, input_ids, scores):
        real_logp = scores.float().log_softmax(-1)
        out = real_logp.clone()
        keep = torch.ones_like(out, dtype=torch.bool)

        # Stage 1: contrast real vs noised image
        if self.beta > 0:
            noisy = model(input_ids=input_ids, attention_mask=torch.ones_like(input_ids),
                          pixel_values=self.noisy_pv, image_grid_thw=self.grid).logits[:, -1].float()
            out = (1 + self.beta) * real_logp - self.beta * noisy.log_softmax(-1)
            keep = real_logp >= real_logp.max(-1, keepdim=True).values + math.log(self.alpha)

        # Stage 2: boost / suppress refusal tokens on the first generated token
        if self.p_unsafe is not None and input_ids.shape[1] == self.prompt_len:
            shift = self.gamma * (2 * self.p_unsafe - 1)
            out[:, REFUSAL_IDS] += shift
            if shift > 0:
                keep[:, REFUSAL_IDS] = True

        return out.masked_fill(~keep, -float("inf"))

## 4. Run one example in three modes

In [ ]:
def generate(image, question, mode="baseline", beta=1.0, max_new_tokens=60):
    inputs = build_inputs(image, question)
    procs, p_unsafe = None, None
    if mode in ("contrastive", "full"):
        if mode == "full":
            p_unsafe = judge_p_unsafe(image, question)
        procs = LogitsProcessorList([MiniSafeCoDe(inputs, beta=beta, p_unsafe=p_unsafe)])
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, logits_processor=procs)
    dt = time.time() - t0
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return dict(mode=mode, text=text, seconds=round(dt, 1),
                peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2),
                p_unsafe=None if p_unsafe is None else round(p_unsafe, 3))

def compare(image, question, beta=1.0):
    display(image.resize((256, int(256 * image.height / image.width))))
    print("Q:", question, "\n")
    for mode in ["baseline", "contrastive", "full"]:
        r = generate(image, question, mode, beta)
        print(f"[{r['mode']}] {r['seconds']}s, {r['peak_gb']} GB, p_unsafe={r['p_unsafe']}\n{r['text']}\n")

## 5. Try it on your own images

Upload two images with the **same question** where the *visual context* changes whether answering is safe (this is the idea behind the MSSBench-style benchmarks the paper uses). For example:
- Question: *"Can I go for a swim here?"* with a calm beach photo vs. a photo with a red danger flag or warning sign.
- Question: *"How do I use this?"* with a kitchen knife in a cooking scene vs. a threatening context.

If you skip the upload, a synthetic placeholder image is used so the notebook still runs end to end.

In [ ]:
def placeholder(text):
    img = Image.new("RGB", (448, 336), (200, 220, 240))
    d = ImageDraw.Draw(img); d.rectangle([20, 120, 428, 216], fill=(255, 255, 255))
    d.text((40, 160), text, fill=(180, 0, 0)); return img

images = []
try:
    from google.colab import files
    print("Upload 1-2 images (or cancel to use placeholders)")
    up = files.upload()
    images = [Image.open(n).convert("RGB") for n in up]
except Exception as e:
    print("No upload, using placeholders:", e)
if not images:
    images = [placeholder("SAFE: calm water, no warnings"), placeholder("DANGER: strong currents, no swimming")]

QUESTION = "Can I go for a swim here?"
for img in images:
    compare(img, QUESTION)
    print("=" * 60)

## 6. Optional: sweep the contrast strength `beta`
A small ablation of your own. Watch how the outputs change as `beta` grows (too large usually hurts fluency).

In [ ]:
for b in [0.0, 0.5, 1.0, 2.0]:
    r = generate(images[0], QUESTION, "full", beta=b)
    print(f"beta={b}: {r['seconds']}s | {r['text'][:150]}")

## Notes and limitations
- Greedy decoding with a short `max_new_tokens`; results on single examples are anecdotal. For real evidence, evaluate on a benchmark such as MSSBench (undersensitivity/oversensitivity) and report refusal rates.
- The noisy-image pass recomputes the full sequence each step (no KV cache), which is simple but slower than an optimized implementation. This overhead is worth measuring and discussing.
- The model judges itself. The paper uses a separate auxiliary judge; swapping it in is a natural extension.
- There are no formal guarantees here. A natural theory question: how far can the contrastive term shift the output distribution (for example in KL) as a function of `beta`, and when does the plausibility mask bound that shift?